# Day 4.8 — Pivotal Exercise: Merge Specialist Findings

This is the day's one hands-on implementation lab. It uses no API key. Try the starter cell first; a fully commented reference solution follows the check so you can compare or catch up.

## Why this mechanism matters

Specialists may overlap, disagree, or fail. Deterministic aggregation makes the supervisor boundary inspectable and avoids spending another model call on rules ordinary code can enforce.

## Contract

Ignore results whose `status` is not `ok`, keep the first copy of each finding `id`, and sort the survivors by descending `severity` then ascending `id`.

Before coding, write one sentence predicting the easiest mistake to make.

In [ ]:
def merge_findings(results):
    """Return one deduplicated, ranked list of findings.

    results -> [{"status": "ok", "findings": [{"id": "F1", "severity": 3}, ...]},
                {"status": "error", "error": "timeout"}, ...]
    """
    # TODO: skip results whose status is not "ok"
    # TODO: keep only the first finding seen for each id
    # TODO: sort by (-severity, id)
    raise NotImplementedError("Complete supervisor merge")

## Behavioural check

Run this after completing the starter cell. If you have not finished, it prints a hint instead of failing. A passing check proves the listed contract examples, not every possible input.

In [ ]:
def run_checks():
    results = [
        {"status": "ok", "findings": [{"id": "F2", "severity": 2}, {"id": "F1", "severity": 3}]},
        {"status": "error", "error": "timeout"},
        {"status": "ok", "findings": [{"id": "F1", "severity": 3}, {"id": "F3", "severity": 1}]},
    ]
    merged = merge_findings(results)
    print("Merged:", [(item["id"], item["severity"]) for item in merged])
    assert [item["id"] for item in merged] == ["F1", "F2", "F3"], "dedupe by id, tolerate the failed specialist"

    # Severity must win over id order: Z1 (severity 4) comes before A9 (severity 1).
    tricky = [{"status": "ok", "findings": [{"id": "A9", "severity": 1}, {"id": "Z1", "severity": 4}]}]
    assert [item["id"] for item in merge_findings(tricky)] == ["Z1", "A9"], "sort by severity first, then id"
    print("PASS: merge is deterministic, deduplicated, ranked, and tolerant of partial failure")

try:
    run_checks()
except NotImplementedError:
    print("Not implemented yet. Complete the starter cell above, or study the reference solution below and re-run this cell.")

## Reference solution

Read this even if your check passed: compare each commented line with your version, then re-run the check cell above.

In [ ]:
# --- Reference solution: read it line by line, then re-run the check cell above ---
def merge_findings(results):
    first_seen = {}                                        # id -> finding (first copy wins)
    for result in results:
        if result.get("status") != "ok":                   # a failed specialist must not erase the others
            print("skipping failed specialist:", result.get("error"))
            continue
        for finding in result["findings"]:
            first_seen.setdefault(finding["id"], finding)  # setdefault keeps the first copy
    return sorted(first_seen.values(), key=lambda f: (-f["severity"], f["id"]))  # high severity first, then id

print("Reference merge_findings defined. Re-run the check cell above to see PASS.")

## Explain

**Why can id-based deduplication still miss semantic duplicates?**

<details><summary>Show answer</summary>

Two specialists can describe the same defect with different ids or different wording. A stable key built from category and line number catches more, but it can also falsely merge two different problems on the same line. Day 4.5 shows why the supervisor keeps both evidence references.

</details>

**Why sort by severity before id?**

<details><summary>Show answer</summary>

The report is read top-down by a busy engineer. Ordering by id would bury a critical finding under cosmetic ones; the id is only a tiebreaker to keep the output deterministic.

</details>